In [ ]:
from dotenv import load_dotenv
load_dotenv()

from youtube_transcript_api import YouTubeTranscriptApi
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain


/Users/rachin/Desktop/GenAI/genai/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
## Get transcript as a paragraph from youtube video


video_id = "qh0FkdgieS8"

yt_api = YouTubeTranscriptApi()
transcript_list = yt_api.fetch(video_id, languages=["en"])

texts = " ".join([sent.text for sent in transcript_list])


In [3]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
docs = text_splitter.create_documents([texts])

In [4]:
llm = ChatOpenAI(model = "gpt-5.4-nano")
embedding = OpenAIEmbeddings(model = "text-embedding-3-large")

In [5]:
vector_store = Chroma.from_documents(docs, embedding)

In [7]:
# vector_store.index_to_docstore_id # for faiss
vector_store.get()["ids"]

['62b4591f-2307-4129-8cf9-4c3ac78e4068',
 '2933b7ea-189a-48c9-b593-ca41a601f4c0',
 'a7bb98a6-3dc6-46e8-ac84-30905d705187',
 'cf6c32d8-0327-45de-9b0e-1cb581fb155d',
 'fe5db7f3-d495-4900-8389-d101445fd1d1',
 '4a2eba03-351c-4344-b1fd-dd5bad9d7bd7',
 '77d177a2-75c6-4c86-ba5e-8462b86a268c',
 '1279524d-5b40-4e9e-bdb6-71b1cbba44f9',
 'ba1115c0-050d-4f5d-bf07-94518b62b537',
 '6d51c226-d87d-4722-929f-e9fd40bd39a2',
 '1ebf6890-9c3e-4b1a-bb8c-2549ff92bd40',
 '95b602ee-d75a-45c3-a82b-605a83868163',
 '9f3639bb-cfca-46a7-b308-4d7df69b6c6e']

In [10]:
vector_store.get(ids = ['1279524d-5b40-4e9e-bdb6-71b1cbba44f9'])

{'ids': ['1279524d-5b40-4e9e-bdb6-71b1cbba44f9'],
 'embeddings': None,
 'documents': ['a neural network created by Google for ImageNet. Karpathy manually labeled around 1,500 difficult\xa0\xa0 images and compared his performance\xa0\ndirectly against the neural network.\xa0 His error rate was 5.1%.\nGoogLeNet’s was 6.8%.\xa0 So the human had technically won.\nBut barely.\xa0 In some instances, the machine\xa0\nactually performed better.\xa0 The neural network had become extraordinarily\xa0\ngood at detecting subtle visual differences across\xa0\xa0 massive datasets, outperforming humans\xa0\nat recognizing things like dog breeds. Karpathy realized something profound:\xa0 “It is clear that humans will soon\xa0\nonly be able to outperform state of\xa0\xa0 the art image classification models by use\xa0\nof significant effort, expertise, and time.” In other words: the machines were\xa0\ncatching up frighteningly fast. And that mattered because neural\xa0\nnetworks were finally becoming\xa0

In [ ]:
prompt = PromptTemplate(
    template = """  
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      Context: {context}
      Question: {input}
     """,
     input_variables = ["context", "input"]
)

In [28]:
document_chain = create_stuff_documents_chain(llm, prompt)

In [29]:
retriever = vector_store.as_retriever(search_type = "similarity", search_kwargs = {"k":3})

In [ ]:
retriever.invoke("who is andrej")
# vector_store.similarity_search("who is andrej")

[Document(id='62b4591f-2307-4129-8cf9-4c3ac78e4068', metadata={}, page_content='At 22 years old, Andrej Karpathy could\xa0\nsolve a Rubik’s Cube in under 16 seconds. That obsession with cracking patterns would\xa0\nserve him well in artificial intelligence. One of the hardest problems is teaching\xa0\nmachines to understand the world around them. Andrej is one of the best computer vision\xa0\npeople in the world. Arguably the best. Okay, thank you. By his early 30s, Karpathy had become one of\xa0\nthe most respected AI researchers in the world.\xa0 So respected that Elon Musk personally\xa0\nrecruited him from OpenAI to lead AI at Tesla.\xa0 In an email to engineer Jim Keller, Musk wrote: Andrej is arguably the #2 guy in\xa0\nthe world in computer vision after\xa0\xa0 Ilya Sutskever. The OpenAI guys are gonna\xa0\nwant to kill me, but it had to be done. Andrej Karpathy was born in Bratislava,\xa0\nSlovakia, on October 23, 1986.\xa0 When he was 15 years old, his family\xa0\nleft their c

In [30]:
from langchain_classic.chains import create_retrieval_chain
chain = create_retrieval_chain(retriever, document_chain)

In [36]:
response = chain.invoke({"input":"who is andrej. tell me in details"})

In [37]:
print(response["answer"])

I know from the transcript that **Andrej Karpathy** is a computer scientist and **one of the best AI / computer vision researchers in the world**.

### Who Andrej Karpathy is
- **Born:** **October 23, 1986**, in **Bratislava, Slovakia**.
- **Moved to Canada:** When he was **15**, his family left their comfortable life and **moved to Toronto** to find better opportunities for him and his sister.
- **Education:** He studied **computer science and physics** at the **University of Toronto**.

### What he originally planned to do
- His original plan was to work in **quantum computing**.
- But during his quantum mechanics classes, he said he realized he **wasn’t having fun**—it felt **too distant, too limiting**, and he “couldn't get [his] hands dirty.”

### How he shifted into AI
- He wanted a field where he could “get hands dirty,” and that field became **artificial intelligence**.
- The transcript describes a moment in a **library**, where he realized he wanted to learn everything from bo

## Another way


In [38]:
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [45]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [46]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'input': RunnablePassthrough()
})

In [47]:
parallel_chain.invoke("who is andrej")

{'context': "At 22 years old, Andrej Karpathy could\xa0\nsolve a Rubik’s Cube in under 16 seconds. That obsession with cracking patterns would\xa0\nserve him well in artificial intelligence. One of the hardest problems is teaching\xa0\nmachines to understand the world around them. Andrej is one of the best computer vision\xa0\npeople in the world. Arguably the best. Okay, thank you. By his early 30s, Karpathy had become one of\xa0\nthe most respected AI researchers in the world.\xa0 So respected that Elon Musk personally\xa0\nrecruited him from OpenAI to lead AI at Tesla.\xa0 In an email to engineer Jim Keller, Musk wrote: Andrej is arguably the #2 guy in\xa0\nthe world in computer vision after\xa0\xa0 Ilya Sutskever. The OpenAI guys are gonna\xa0\nwant to kill me, but it had to be done. Andrej Karpathy was born in Bratislava,\xa0\nSlovakia, on October 23, 1986.\xa0 When he was 15 years old, his family\xa0\nleft their comfortable life behind\xa0\xa0 and moved to Toronto in search of be

In [48]:
parser = StrOutputParser()

In [49]:
main_chain = parallel_chain | prompt | llm | parser

In [50]:
response = main_chain.invoke("who is andrej")
print(response)

From the transcript, **Andrej Karpathy** is a highly respected **AI researcher and one of the best computer vision experts in the world**—recruited by **Elon Musk** to lead AI at **Tesla**.
